# S4: Hyperparameter Sensitivity Analysis

**Leaf-JEPA IRP** | Stage 5 — PEFT Adaptation Experiments

**Research Role:** Deepens RQ2 — reveals the accuracy-vs-capacity curve and diminishing returns.

**Key Output:** Pareto frontier figure — the definitive answer to RQ2.

**Prerequisite:** Run S1 first.

## Imports & Configuration

In [1]:
import sys
import json
from pathlib import Path
from collections import defaultdict

PROJECT_ROOT = Path("..").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt

from stage5_peft_adaptation_experiments.config_stage5 import *
from stage5_peft_adaptation_experiments.peft_utils import (
    set_seed, get_device, train_peft, aggregate_seed_results,
    save_results, load_results, plot_pareto
)

verify_config()
device = get_device()

SET4_DIR = PEFT_DIR / "set4"
SET4_DIR.mkdir(parents=True, exist_ok=True)

Stage 5 Configuration Verification
  ✅  PlantVillage root exists
  ✅  PlantDoc root exists
  ✅  Splits directory exists
  ✅  Normalisation stats exist
  ✅  NORM_MEAN matches JSON
  ✅  NORM_STD matches JSON
  ✅  I-JEPA checkpoint exists
  ❌  Leaf-JEPA checkpoint exists
       → Run Stage 4 first, then update LEAF_JEPA_CHECKPOINT in config_stage5.py
  ✅  Baselines directory exists
  ✅  WANDB_ENTITY set

  9/10 checks passed
  ⚠️  Fix failing checks before running experiments!
Using GPU: NVIDIA GeForce RTX 5060 Laptop GPU
VRAM: 8.5 GB


## Extended LoRA Rank Sweep

r ∈ {1, 2, 4, 8, 16, 32} — 6 values × 3 seeds = 18 runs

In [ ]:
lora_results = []

for r in LORA_RANKS_EXTENDED:
    for seed in SUBSET_SEEDS:
        run_name = f"S5-LoRA-r{r}-frac1.00-seed{seed}"
        print(f"\n  {run_name}")
        
        result = train_peft(
            method="lora",
            checkpoint_path=LEAF_JEPA_CHECKPOINT,
            pv_root=PV_ROOT,
            splits_dir=SPLITS_DIR,
            norm_mean=NORM_MEAN,
            norm_std=NORM_STD,
            model_name=VIT_MODEL_NAME,
            embed_dim=VIT_EMBED_DIM,
            num_classes=NUM_CLASSES,
            fraction=1.0,
            seed=seed,
            lr=BEST_LR["lora"],
            batch_size=BATCH_SIZE,
            max_epochs=MAX_EPOCHS,
            patience=EARLY_STOP_PATIENCE,
            weight_decay=WEIGHT_DECAY,
            warmup_fraction=WARMUP_FRACTION,
            use_amp=USE_AMP,
            gradient_clip=GRADIENT_CLIP,
            num_workers=NUM_WORKERS,
            class_weights_path=CLASS_WEIGHTS_PATH,
            save_dir=SET4_DIR,
            run_name=run_name,
            wandb_project=WANDB_PROJECT,
            wandb_entity=WANDB_ENTITY,
            wandb_group=WANDB_GROUPS["set4"],
            rank=r,
        )
        result["hp_tag"] = f"r{r}"
        lora_results.append(result)

save_results(lora_results, SET4_DIR / "lora_extended_sweep.json")
print(f"\n✅ LoRA extended sweep complete: {len(lora_results)} runs")

## Extended Adapter Dimension Sweep

dim ∈ {4, 8, 16, 32, 64, 128} — 6 values × 3 seeds = 18 runs

In [ ]:
adapter_results = []

for d in ADAPTER_DIMS_EXTENDED:
    for seed in SUBSET_SEEDS:
        run_name = f"S5-Adapter-d{d}-frac1.00-seed{seed}"
        print(f"\n  {run_name}")
        
        result = train_peft(
            method="adapter",
            checkpoint_path=LEAF_JEPA_CHECKPOINT,
            pv_root=PV_ROOT,
            splits_dir=SPLITS_DIR,
            norm_mean=NORM_MEAN,
            norm_std=NORM_STD,
            model_name=VIT_MODEL_NAME,
            embed_dim=VIT_EMBED_DIM,
            num_classes=NUM_CLASSES,
            fraction=1.0,
            seed=seed,
            lr=BEST_LR["adapter"],
            batch_size=BATCH_SIZE,
            max_epochs=MAX_EPOCHS,
            patience=EARLY_STOP_PATIENCE,
            weight_decay=WEIGHT_DECAY,
            warmup_fraction=WARMUP_FRACTION,
            use_amp=USE_AMP,
            gradient_clip=GRADIENT_CLIP,
            num_workers=NUM_WORKERS,
            class_weights_path=CLASS_WEIGHTS_PATH,
            save_dir=SET4_DIR,
            run_name=run_name,
            wandb_project=WANDB_PROJECT,
            wandb_entity=WANDB_ENTITY,
            wandb_group=WANDB_GROUPS["set4"],
            bottleneck_dim=d,
        )
        result["hp_tag"] = f"d{d}"
        adapter_results.append(result)

save_results(adapter_results, SET4_DIR / "adapter_extended_sweep.json")
print(f"\n✅ Adapter extended sweep complete: {len(adapter_results)} runs")

## Pareto Frontier Plot

In [ ]:
all_set4 = lora_results + adapter_results

# Also load Set 1 for other methods
set1_results = load_results(PEFT_DIR / "set1_method_comparison.json")

# Build Pareto data
pareto_data = []
for res in all_set4 + set1_results:
    pareto_data.append({
        "method": res["method"],
        "trainable_params": res["param_count"]["trainable"],
        "macro_f1": res["results"]["test_macro_f1"],
        "label": f"{res['method']} ({res.get('hp_tag', '')})",
    })

plot_pareto(
    pareto_data,
    save_path=FIGURES_DIR / "set4_pareto_frontier.png",
    title="Pareto Frontier: Macro-F1 vs Trainable Parameters",
)
print(f"Saved: {FIGURES_DIR / 'set4_pareto_frontier.png'}")

## Diminishing Returns Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sweep_data = [
    (lora_results, "rank", "LoRA Rank (r)", LORA_RANKS_EXTENDED),
    (adapter_results, "bottleneck_dim", "Adapter Dim", ADAPTER_DIMS_EXTENDED),
]

for ax, (results, param_name, xlabel, hp_values) in zip(axes, sweep_data):
    grouped = defaultdict(list)
    for res in results:
        hp_val = res["hyperparams"][param_name]
        grouped[hp_val].append(res["results"]["test_macro_f1"])
    
    hp_vals = sorted(grouped.keys())
    means = [np.mean(grouped[v]) for v in hp_vals]
    stds = [np.std(grouped[v]) for v in hp_vals]
    
    ax.errorbar(hp_vals, means, yerr=stds, fmt="o-", capsize=5,
                linewidth=2, markersize=8, color="#2c3e50")
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel("Macro-F1", fontsize=12)
    ax.set_title(f"Diminishing Returns: {xlabel}", fontsize=13)
    ax.grid(True, alpha=0.3)
    
    # Annotate marginal gains
    for i in range(1, len(means)):
        gain = means[i] - means[i - 1]
        colour = "#27ae60" if gain > 0.005 else "#e74c3c" if gain < 0 else "#7f8c8d"
        ax.annotate(
            f"{gain:+.3f}",
            xy=(hp_vals[i], means[i]),
            textcoords="offset points",
            xytext=(10, 5),
            fontsize=8,
            color=colour,
        )

plt.tight_layout()
plt.savefig(FIGURES_DIR / "set4_diminishing_returns.png", dpi=150)
plt.show()

save_results(
    {"lora": lora_results, "adapter": adapter_results},
    PEFT_DIR / "set4_sensitivity.json",
)
print("\n✅ Set 4 complete.")